# Verify Persisted Fact-Grounding Checkpoint

This notebook does not train. It recomputes predictions for the official MuSiQue dev split from the persisted checkpoint in original row order, reports whether the old prediction file matches, then reloads the checkpoint and compares a fixed sample. It logs every ten inference batches and saves diagnostics before checking the reload.


In [ ]:
import sys,subprocess,json,time,os
from pathlib import Path
def log(stage,status,**details):
    print(json.dumps({'time':time.strftime('%Y-%m-%d %H:%M:%S'),'checkpoint':stage,'status':status,**details},default=str),flush=True)
log('00_environment','begin')
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers==4.57.1','sentencepiece','protobuf','scikit-learn'])
import numpy as np,pandas as pd,torch,transformers
from transformers import AutoTokenizer,AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score,confusion_matrix,precision_recall_fscore_support
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type=='cpu': torch.set_num_threads(min(4,os.cpu_count() or 1))
log('00_environment','complete',torch=torch.__version__,transformers=transformers.__version__,device=str(device),gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
log('01_find_persisted_checkpoint','begin')
matches=[]
for candidate in Path('/kaggle/input').iterdir():
    configs=list(candidate.rglob('config.json'))
    checkpoints=[p.parent for p in configs if (p.parent/'model.safetensors').exists() or (p.parent/'pytorch_model.bin').exists()]
    preferred=next((p for p in checkpoints if p.name=='deberta_fact_grounding_checkpoint'),None)
    checkpoint=preferred or (checkpoints[0] if checkpoints else None)
    pred_files=list(candidate.rglob('in_domain_predictions.csv'))
    metric_files=list(candidate.rglob('in_domain_metrics.json'))
    if checkpoint is not None and pred_files and metric_files: matches.append((candidate,checkpoint,pred_files[0],metric_files[0]))
if len(matches)!=1: raise FileNotFoundError(f'Expected one attached training-output/checkpoint input with model weights and in-domain metrics, found {[m[0].name for m in matches]}')
root,checkpoint,pred_file,metric_file=matches[0]
pred=pd.read_csv(pred_file); metrics=json.loads(metric_file.read_text(encoding='utf-8'))
required={'text','label','prob_answerable','prediction'}
if not required<=set(pred.columns): raise ValueError(f'Missing prediction columns: {required-set(pred.columns)}')
assert len(pred)==6404, f'Expected 6404 saved dev predictions, got {len(pred)}'
dev_files=list(root.rglob('deberta_fg_dev.csv'))
if len(dev_files)!=1: raise FileNotFoundError(f'Expected one official dev split in {root}, found {dev_files}')
dev=pd.read_csv(dev_files[0],dtype={'text':str,'label':int})
if not {'text','label'}<=set(dev.columns): raise ValueError(f'Official dev split is missing text/label: {dev.columns.tolist()}')
if len(dev)!=6404: raise ValueError(f'Expected 6404 official dev rows, found {len(dev)}')
same_source_order=(len(pred)==len(dev) and pred.text.fillna('').equals(dev.text.fillna('')) and np.array_equal(pred.label.to_numpy(dtype=int),dev.label.to_numpy(dtype=int)))
log('01_find_persisted_checkpoint','complete',input=root.name,checkpoint=str(checkpoint),checkpoint_files=sorted(p.name for p in checkpoint.iterdir()),saved_predictions=len(pred),official_dev_rows=len(dev),saved_rows_match_dev_order=same_source_order,saved_metrics=metrics)

In [ ]:
log('02_load_checkpoint','begin',checkpoint=str(checkpoint),device=str(device))
stage=time.perf_counter(); tokenizer=AutoTokenizer.from_pretrained(checkpoint)
log('02a_load_tokenizer','complete',seconds=round(time.perf_counter()-stage,2))
stage=time.perf_counter(); model=AutoModelForSequenceClassification.from_pretrained(checkpoint).to(device).eval()
if model.config.label2id.get('ANSWERABLE')!=1 or model.config.label2id.get('NOT-ANSWERABLE')!=0: raise ValueError(f'Unexpected label mapping: {model.config.label2id}')
log('02b_load_model','complete',seconds=round(time.perf_counter()-stage,2),parameters=sum(p.numel() for p in model.parameters()))
batch_size=8 if device.type=='cuda' else 4
dev_text=dev.text.fillna('').astype(str).tolist(); probabilities=[]; started=time.perf_counter()
total_batches=(len(dev_text)+batch_size-1)//batch_size
for batch_no,start in enumerate(range(0,len(dev_text),batch_size),1):
    batch_texts=dev_text[start:start+batch_size]
    inputs=tokenizer(batch_texts,max_length=512,padding=True,pad_to_multiple_of=8 if device.type=='cuda' else None,truncation=True,return_tensors='pt').to(device)
    with torch.inference_mode(): probs=torch.softmax(model(**inputs).logits.float(),dim=-1)[:,1].cpu().numpy()
    probabilities.extend(probs.tolist())
    if batch_no%10==0 or batch_no==total_batches: log('03_full_dev_inference','progress',completed_rows=min(start+len(batch_texts),len(dev)),total_rows=len(dev),batch=batch_no,total_batches=total_batches,elapsed_seconds=round(time.perf_counter()-started,1))
fresh=np.asarray(probabilities,dtype=float); fresh_labels=(fresh>=0.5).astype(int); gold=dev.label.to_numpy(dtype=int)
precision,recall,f1,_=precision_recall_fscore_support(gold,fresh_labels,average='binary',pos_label=1,zero_division=0)
fresh_metrics={'dataset':'MuSiQue dev','n':len(dev),'n_positive':int(gold.sum()),'n_negative':int((gold==0).sum()),'threshold':0.5,'accuracy':float(accuracy_score(gold,fresh_labels)),'precision':float(precision),'recall':float(recall),'f1':float(f1),'confusion_matrix_labels_0_1':confusion_matrix(gold,fresh_labels,labels=[0,1]).tolist()}
fresh_predictions=dev[['text','label']].copy(); fresh_predictions['prob_answerable']=fresh; fresh_predictions['prediction']=fresh_labels
fresh_predictions.to_csv('/kaggle/working/recomputed_in_domain_predictions.csv',index=False)
Path('/kaggle/working/recomputed_in_domain_metrics.json').write_text(json.dumps(fresh_metrics,indent=2),encoding='utf-8')
saved_text_match=(len(pred)==len(dev) and pred.text.fillna('').equals(dev.text.fillna('')))
saved_label_match=(len(pred)==len(dev) and np.array_equal(pred.label.to_numpy(dtype=int),gold))
saved_prob=pred.prob_answerable.to_numpy(dtype=float) if saved_text_match and saved_label_match else np.full(len(dev),np.nan)
saved_diff=np.abs(fresh-saved_prob) if np.isfinite(saved_prob).all() else np.array([],dtype=float)
indices=np.linspace(0,len(dev)-1,num=min(32,len(dev)),dtype=int).tolist()
pd.DataFrame({'sample_index':indices,'label':gold[indices],'saved_prediction':pred.prediction.to_numpy(dtype=int)[indices] if saved_label_match else np.full(len(indices),-1),'fresh_prediction':fresh_labels[indices],'saved_probability':saved_prob[indices] if len(saved_prob) else np.full(len(indices),np.nan),'fresh_probability':fresh[indices],'abs_probability_diff':saved_diff[indices] if len(saved_diff) else np.full(len(indices),np.nan)}).to_csv('/kaggle/working/checkpoint_reload_samples.csv',index=False)
saved_metrics_delta={key:float(fresh_metrics[key]-metrics[key]) for key in ['accuracy','precision','recall','f1'] if key in metrics}
reload_indices=np.array(sorted(set([0,len(dev)//2,len(dev)-1])),dtype=int); reference=fresh[reload_indices].copy(); del model
if device.type=='cuda': torch.cuda.empty_cache()
log('04_reload_checkpoint','begin',sample_indices=reload_indices.tolist())
stage=time.perf_counter(); reload_model=AutoModelForSequenceClassification.from_pretrained(checkpoint).to(device).eval(); reload_probs=[]
with torch.inference_mode():
    for idx in reload_indices:
        inputs=tokenizer(dev_text[idx],max_length=512,padding='max_length',truncation=True,return_tensors='pt').to(device)
        reload_probs.append(float(torch.softmax(reload_model(**inputs).logits.float(),dim=-1)[0,1].cpu()))
reload_diff=float(np.max(np.abs(reference-np.asarray(reload_probs))))
report={'checkpoint_dir':str(checkpoint),'device':str(device),'sample_count':len(indices),'full_dev_inference_seconds':round(time.perf_counter()-started,2),'fresh_dev_metrics':fresh_metrics,'saved_dev_metrics':metrics,'fresh_minus_saved_metric_delta':saved_metrics_delta,'saved_text_order_matches_dev':saved_text_match,'saved_labels_match_dev':saved_label_match,'saved_predictions_match_fresh':bool(saved_text_match and saved_label_match and np.array_equal(pred.prediction.to_numpy(dtype=int),fresh_labels)),'max_abs_saved_probability_diff':float(np.max(saved_diff)) if len(saved_diff) else None,'reload_sample_indices':reload_indices.tolist(),'reload_max_abs_probability_diff':reload_diff,'reload_tolerance':0.001,'checkpoint_reload_pass':bool(reload_diff<=0.001),'recomputed_predictions_path':'/kaggle/working/recomputed_in_domain_predictions.csv','recomputed_metrics_path':'/kaggle/working/recomputed_in_domain_metrics.json','sample_diagnostics_path':'/kaggle/working/checkpoint_reload_samples.csv'}
report_path=Path('/kaggle/working/checkpoint_reload_report.json'); report_path.write_text(json.dumps(report,indent=2),encoding='utf-8')
log('05_report_saved','complete',report_path=str(report_path),metrics_path='/kaggle/working/recomputed_in_domain_metrics.json',predictions_path='/kaggle/working/recomputed_in_domain_predictions.csv',checkpoint_reload_pass=report['checkpoint_reload_pass'],fresh_metrics=fresh_metrics,saved_predictions_match_fresh=report['saved_predictions_match_fresh'],max_abs_saved_probability_diff=report['max_abs_saved_probability_diff'],reload_max_abs_probability_diff=reload_diff)
if not report['checkpoint_reload_pass']: raise RuntimeError(f'Fresh checkpoint reload changed sampled probabilities; report saved at {report_path}: {report}')
log('06_run_complete','complete',report_path=str(report_path))